# MJO 6: Calculate Principle Component from EOF Analysis

Via `mjo_EOF_cal.ncl`

In [1]:
# # getenv == os.environ
import os

CASENAME = "QBOi.EXP1.AMIP.001"
startdate = '19790101'
enddate = '19811231'

# /glade/u/home/bundy/mdtf/MDTF_3_main/MDTF-diagnostics.blocking_notebook/diagnostics/MJO_suite/MJO_driver.py
#DATADIR = "/glade/u/home/bundy/diag/mdtf/inputdata/model/QBOi.EXP1.AMIP.001/"
WORK_DIR = os.getcwd()
DATADIR = "/data/"

U200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U200.day.nc"
V200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V200.day.nc"
U850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U850.day.nc"
V850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V850.day.nc"
RLUT_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.FLUT.day.nc"
PR_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.PRECT.day.nc"

lev_coord = "lev"
lat_coord = "lat"
lon_coord = "lon"
time_coord = "time"
pr_var = "PRECT"
rlut_var = "FLUT"

u200_var = "U200"
v200_var = "V200"

wk_dir = WORK_DIR + "/model/"

# setup data directory, if does not already exist
#if not os.path.exists(DATADIR): os.makedirs(DATADIR)

In [10]:
# Combined EOFs

file_dir = WORK_DIR + "/model/"
pltDir = file_dir + "/PS/" # plot directory
pltType = "ps"
pltName = "mjoclivar"


neof = 2
latS = -20
latN = 20

anom_dir = WORK_DIR + DATADIR + "anomaly/"
fileolr = anom_dir + CASENAME + ".flut.day.anom.nc"
fileu850 = anom_dir + CASENAME + ".u850.day.anom.nc"
fileu200 = anom_dir + CASENAME + ".u200.day.anom.nc"

In [13]:
import xarray as xr
file_olr = xr.open_dataset(fileolr)
file_olr

<xarray.Dataset> Size: 565MB
Dimensions:  (time: 2555, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 20kB ...
    FLUT     (time, lat, lon) float32 565MB ...
Attributes:
    history:  Thu Mar  6 13:31:31 2025: ncatted -a cell_methods,FLUT,m,c,time...
    NCO:      netCDF Operators version 5.3.1 (Homepage = http://nco.sf.net, C...

In [14]:
# get indicies corresponding to the start/end times
TIME = file_olr["time"]
ymdStrt = file_olr["time"][file_olr.time.argmin()]
ymdLast = file_olr["time"][file_olr.time.argmax()]
yrStrt = ymdStrt["time"].item().year
yrLast = ymdLast["time"].item().year

### Fixed Lanczos Filter Weights

Create bandpass filter from weights

See: [Lanczos Filter Weights](https://www.ncl.ucar.edu/Applications/mjoclivar.shtml)

```
When needed, the weights for the suggested 20-100 day bandpass Lanczos filter are generated 'on-the-fly' using:

  ihp      = 2                             ; bpf=>band pass filter
  nWgt     = 201
  sigma    = 1.0                           ; Lanczos sigma
  fca      = 1./100.
  fcb      = 1./20.
  wgt      = filwgts_lanczos (nWgt, ihp, fca, fcb, sigma )

```

See `generate_lanczos_filter_weights.ncl` which generates `lanczos_filter_weights_output.txt`

In [3]:
import numpy as np
def lanczos_filter_weights(nwt=None, ihp=None, fca=None, fcb=None, nsigma=None):
    # nwt = scale indicating the total number of weights (must be an odd number, nwt >= 3).The more weights, the better the filter, but greater loss of data
    # ihp = scale indicating the low-pass filter
    # fca = scale indicating the cut-off frequency of the ideal high or low-pass filter (0 < fca < 0.5)
    # fcb = scale used only when band-pass filter is desired, second cut-off frequency (fca < fcb < 0.5)
    # scale indicating the power of sigma factor (nsigma >= 0), ngima = 1 is common
    # returns: a symmetrical set of weights
    ncl_output_weights = np.loadtxt("lanczos_filter_weights_output.txt", delimiter=",", skiprows=17)
    return ncl_output_weights 

In [4]:
weights = lanczos_filter_weights()
weights

array([ 1.933179e-11, -1.602059e-05, -4.602891e-05, -8.413403e-05,
       -1.211932e-04, -1.459612e-04, -1.466584e-04, -1.127676e-04,
       -3.682710e-05,  8.402883e-05,  2.470150e-04,  4.433134e-04,
        6.584777e-04,  8.736432e-04,  1.067462e-03,  1.218581e-03,
        1.308390e-03,  1.323730e-03,  1.259200e-03,  1.118744e-03,
        9.162520e-04,  6.749944e-04,  4.258647e-04,  2.045088e-04,
        4.758891e-05, -1.146049e-05,  5.326822e-05,  2.562978e-04,
        5.978383e-04,  1.062210e-03,  1.617924e-03,  2.219547e-03,
        2.811319e-03,  3.332239e-03,  3.722208e-03,  3.928611e-03,
        3.912660e-03,  3.654768e-03,  3.158300e-03,  2.451133e-03,
        1.584706e-03,  6.304432e-04, -3.262688e-04, -1.194086e-03,
       -1.884996e-03, -2.323995e-03, -2.458149e-03, -2.264029e-03,
       -1.752608e-03, -9.709108e-04, -1.243057e-09,  1.050766e-03,
        2.052896e-03,  2.870605e-03,  3.374352e-03,  3.454755e-03,
        3.035499e-03,  2.083817e-03,  6.173067e-04, -1.293891e

In [15]:
# read anomalies

# RLUT
print(fileolr)
work = file_olr["FLUT"].sel(lat=slice(latS, latN))
rightmost_dim = list(work.sizes.keys())[-1]
#print(f"rightmost dimension: {rightmost_dim}\n")
OLR = work.mean(dim=rightmost_dim) # dim_avg_wrap

/home/cs/Github/geocat-research/mjo/data/anomaly/QBOi.EXP1.AMIP.001.flut.day.anom.nc


In [16]:
# read anomalies

# U850
print(fileu850)
file_u850 = xr.open_dataset(fileu850)
work = file_u850["U850"].sel(lat=slice(latS, latN))
rightmost_dim = list(work.sizes.keys())[-1]
#print(f"rightmost dimension: {rightmost_dim}\n")
U850 = work.mean(dim=rightmost_dim) # dim_avg_wrap

/home/cs/Github/geocat-research/mjo/data/anomaly/QBOi.EXP1.AMIP.001.u850.day.anom.nc


In [17]:
# read anomalies

# U200
print(fileu200)
file_u200 = xr.open_dataset(fileu200)
work = file_u200["U200"].sel(lat=slice(latS, latN))
rightmost_dim = list(work.sizes.keys())[-1]
#print(f"rightmost dimension: {rightmost_dim}\n")
U200 = work.mean(dim=rightmost_dim) # dim_avg_wrap

/home/cs/Github/geocat-research/mjo/data/anomaly/QBOi.EXP1.AMIP.001.u200.day.anom.nc


In [18]:
dimw = work.shape
print(dimw)
ntim = dimw[0]
nlat = dimw[1]
mlon = dimw[2]

(2555, 42, 288)


In [19]:
lon = file_u200["lon"]
time = file_u200["time"]

#TODO: L88

In [20]:
# Apply the band pass filter to the original anomalies

# wgt_runave_wrap: calculates a weighted running average on the rightmost dimension and retains metadata
def wgt_runave_wrap(dataset):
    print("Calculating weighted running average...")
    rightmost_dim = list(dataset.sizes.keys())[-1]
    weighted_running_average = dataset.rolling(time=len(weights), center=True).mean().dropna("time")
    print(weighted_running_average)
    return weighted_running_average

rlut = wgt_runave_wrap(OLR)
u850 = wgt_runave_wrap(U850)
u200 = wgt_runave_wrap(U200)

Calculating weighted running average...
<xarray.DataArray 'FLUT' (time: 2355, lat: 42)> Size: 396kB
array([[-0.59653676, -0.6743853 , -0.8312625 , ...,  1.033094  ,
         0.830143  ,  0.70461375],
       [-0.6342632 , -0.71644044, -0.87524205, ...,  1.0339069 ,
         0.8300379 ,  0.70379114],
       [-0.6520694 , -0.74041814, -0.9052075 , ...,  1.0651143 ,
         0.8510699 ,  0.70786697],
       ...,
       [ 0.03020744,  0.14304619,  0.2949625 , ..., -0.48146763,
        -1.0451391 , -1.469548  ],
       [ 0.05949442,  0.17251283,  0.31980973, ..., -0.5136553 ,
        -1.081141  , -1.5079143 ],
       [ 0.10944816,  0.21853478,  0.35612836, ..., -0.5314985 ,
        -1.1199778 , -1.5591122 ]], shape=(2355, 42), dtype=float32)
Coordinates:
  * time     (time) object 19kB 1975-04-11 00:00:00 ... 1981-09-22 00:00:00
  * lat      (lat) float64 336B -19.32 -18.38 -17.43 ... 17.43 18.38 19.32
Calculating weighted running average...
<xarray.DataArray 'U850' (time: 2355, lat: 42)> Si

In [21]:
# TODO: remove means of band pass series (not necessary)
# L100-102

In [22]:
# Compute the temporal variance

# dim_variance_Wrap -> Computes unbiased estimates of the variance of a variable's rightmost dimension
def dim_variance_wrap(dataset):
    print("Computing variance...")
    rightmost_dim = list(dataset.sizes.keys())[-1]
    variance = dataset.var(dim=rightmost_dim)
    print(variance)
    return variance
    
var_rlut = dim_variance_wrap(rlut)
var_u850 = dim_variance_wrap(u850)
var_u200 = dim_variance_wrap(u200)

Computing variance...
<xarray.DataArray 'FLUT' (time: 2355)> Size: 9kB
array([7.260028 , 7.483185 , 7.588971 , ..., 2.071576 , 2.0184197,
       1.9826462], shape=(2355,), dtype=float32)
Coordinates:
  * time     (time) object 19kB 1975-04-11 00:00:00 ... 1981-09-22 00:00:00
Computing variance...
<xarray.DataArray 'U850' (time: 2355)> Size: 9kB
array([0.07964514, 0.079275  , 0.0782259 , ..., 0.02192665, 0.0210259 ,
       0.02009254], shape=(2355,), dtype=float32)
Coordinates:
  * time     (time) object 19kB 1975-04-11 00:00:00 ... 1981-09-22 00:00:00
Computing variance...
<xarray.DataArray 'U200' (time: 2355)> Size: 9kB
array([0.10816468, 0.11388068, 0.11798826, ..., 0.294593  , 0.30529326,
       0.31182164], shape=(2355,), dtype=float32)
Coordinates:
  * time     (time) object 19kB 1975-04-11 00:00:00 ... 1981-09-22 00:00:00


In [23]:
# Compute the zonal mean of the temporal variance

def dim_avg_wrap(dataset):
    print("Computing average...")
    rightmost_dim = list(dataset.sizes.keys())[-1]
    avg = dataset.mean(dim=rightmost_dim)
    print(avg)
    return avg

zavg_var_rlut = dim_avg_wrap(var_rlut)
zavg_var_u850 = dim_avg_wrap(var_u850)
zavg_var_u200 = dim_avg_wrap(var_u200)

Computing average...
<xarray.DataArray 'FLUT' ()> Size: 8B
array(2.59999108)
Computing average...
<xarray.DataArray 'U850' ()> Size: 8B
array(0.02625952)
Computing average...
<xarray.DataArray 'U200' ()> Size: 8B
array(0.3113676)


In [ ]:
# Noramlize by sqrt(avg_var)

rlut = rlut/np.sqrt(zavg_var_rlut)
u850 = u850/np.sqrt(zavg_var_u850)
u200 = u200/np.sqrt(zavg_var_u200)

print(rlut)
print(u850)
print(u200)

In [24]:
# Combine the noramlized data into variable
cdata = xr.DataArray(np.full((3*mlon, ntim), np.nan, dtype=np.float32))
print(cdata.shape)
print(cdata)

for m1 in range(mlon-1):
    print(cdata[m1,:])
    print(rlut[m1,:])
    print("\n")
    print(cdata.values[m1,:].shape)
    print(rlut[m1,:].shape)
    cdata.values[m1,:] = rlut.values[m1,:]
    #cdata[m1+mlon,:] = u850[m1,:]
    #cdata[m1+2*mlon,:] = u200[m1,:]

(864, 2555)
<xarray.DataArray (dim_0: 864, dim_1: 2555)> Size: 9MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(864, 2555), dtype=float32)
Dimensions without coordinates: dim_0, dim_1
<xarray.DataArray (dim_1: 2555)> Size: 10kB
array([nan, nan, nan, ..., nan, nan, nan], shape=(2555,), dtype=float32)
Dimensions without coordinates: dim_1
<xarray.DataArray 'FLUT' (lat: 42)> Size: 168B
array([-0.59653676, -0.6743853 , -0.8312625 , -1.0711544 , -1.4363431 ,
       -1.8623284 , -2.2115288 , -2.5600889 , -2.8536096 , -3.2001624 ,
       -3.5572708 , -3.7521248 , -3.7421608 , -3.6957388 , -3.6063123 ,
       -3.2707465 , -2.5177383 , -1.5356828 , -0.27915347,  0.59173197,
        1.1372918 ,  1.2082767 ,  1.0202429 ,  0.9609505 ,  0.85701597,
        1

ValueError: could not broadcast input array from shape (42,) into shape (2555,)